In [0]:
# Install required libraries
%pip install optuna scikit-learn category-encoders

dbutils.library.restartPython()

In [0]:
%pip install category_encoders

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession

# Load the dataset
df_spark = spark.table("workspace.default.train_delay_data")
df = df_spark.toPandas()

# Display basic information
print("Dataset shape:", df.shape)
print("\nColumn names and data types:")
print(df.dtypes)
print("\nFirst few rows:")
display(df.head())

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Statistical summary
print("\nStatistical summary:")
display(df.describe())

# Feature type table
feature_types = pd.DataFrame({
    'Feature': df.columns,
    'Type': [
        'Numeric (integer)',  # Distance Between Stations (km)
        'Categorical/Nominal',  # Weather Conditions
        'Categorical/Nominal',  # Day of the Week
        'Categorical/Nominal',  # Time of Day
        'Categorical/Nominal',  # Train Type
        'Numeric (integer) - Target',  # Historical Delay (min)
        'Categorical/Ordinal'  # Route Congestion
    ]
})
print("\nFeature Types:")
display(feature_types)

In [0]:
# Set plot style
sns.set_style('whitegrid')

# 1. Correlation between Distance and Delay
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Scatter plot: Distance vs Delay
axes[0, 0].scatter(df['Distance Between Stations (km)'], df['Historical Delay (min)'], alpha=0.6)
axes[0, 0].set_xlabel('Distance Between Stations (km)')
axes[0, 0].set_ylabel('Historical Delay (min)')
axes[0, 0].set_title('Distance vs Delay')

# 2. Weather Conditions vs Delay
sns.boxplot(data=df, x='Weather Conditions', y='Historical Delay (min)', ax=axes[0, 1])
axes[0, 1].set_title('Weather Conditions vs Delay')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Train Type vs Delay
sns.boxplot(data=df, x='Train Type', y='Historical Delay (min)', ax=axes[0, 2])
axes[0, 2].set_title('Train Type vs Delay')
axes[0, 2].tick_params(axis='x', rotation=45)

# 4. Route Congestion vs Delay
sns.boxplot(data=df, x='Route Congestion', y='Historical Delay (min)', ax=axes[1, 0])
axes[1, 0].set_title('Route Congestion vs Delay')
axes[1, 0].tick_params(axis='x', rotation=45)

# 5. Day of Week vs Delay
sns.boxplot(data=df, x='Day of the Week', y='Historical Delay (min)', ax=axes[1, 1])
axes[1, 1].set_title('Day of Week vs Delay')
axes[1, 1].tick_params(axis='x', rotation=45)

# 6. Time of Day vs Delay
sns.boxplot(data=df, x='Time of Day', y='Historical Delay (min)', ax=axes[1, 2])
axes[1, 2].set_title('Time of Day vs Delay')
axes[1, 2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print category distributions
print("\nCategorical Feature Distributions:")
for col in ['Weather Conditions', 'Train Type', 'Route Congestion', 'Day of the Week', 'Time of Day']:
    print(f"\n{col}:")
    print(df[col].value_counts())

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from category_encoders import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

# Separate features and target
X = df.drop('Historical Delay (min)', axis=1)
y = df['Historical Delay (min)']

# Define categorical and numerical columns
categorical_cols = ['Weather Conditions', 'Day of the Week', 'Time of Day', 'Train Type', 'Route Congestion']
numerical_cols = ['Distance Between Stations (km)']

# Split the data (70% train, 15% validation, 15% test)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)  # 0.1765 * 0.85 ≈ 0.15

print(f"Training set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set size: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print("\nSplit justification: 70% training provides sufficient data for learning,")
print("15% validation for hyperparameter tuning, and 15% test for final evaluation.")

# One-hot encode categorical variables
encoder = OneHotEncoder(cols=categorical_cols, use_cat_names=True)
X_train_encoded = encoder.fit_transform(X_train)
X_val_encoded = encoder.transform(X_val)
X_test_encoded = encoder.transform(X_test)

# Scale numerical features
scaler = StandardScaler()
X_train_encoded[numerical_cols] = scaler.fit_transform(X_train_encoded[numerical_cols])
X_val_encoded[numerical_cols] = scaler.transform(X_val_encoded[numerical_cols])
X_test_encoded[numerical_cols] = scaler.transform(X_test_encoded[numerical_cols])

print(f"\nOriginal features: {X.shape[1]}")
print(f"Features after one-hot encoding: {X_train_encoded.shape[1]}")
print("\nEncoded feature names:")
print(list(X_train_encoded.columns))

In [0]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import mlflow
import mlflow.sklearn
import optuna
import time

# Set MLflow experiment
mlflow.set_experiment("/Users/che240008018@iiti.ac.in/train-delay-prediction")

# Define the objective function for Optuna
def objective(trial):
    # Suggest hyperparameters
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state': 42
    }
    
    # Train model
    model = GradientBoostingRegressor(**params)
    model.fit(X_train_encoded, y_train)
    
    # Validate
    y_val_pred = model.predict(X_val_encoded)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    
    return rmse

# Run hyperparameter optimization
print("Starting hyperparameter optimization with Optuna...")
start_time = time.time()

study = optuna.create_study(direction='minimize', study_name='train_delay_gbt')
study.optimize(objective, n_trials=50, show_progress_bar=True)

optimization_time = time.time() - start_time

print(f"\nOptimization completed in {optimization_time:.2f} seconds")
print(f"Best RMSE: {study.best_value:.4f}")
print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

In [0]:
from mlflow.models import infer_signature

# Train final model with best parameters
with mlflow.start_run(run_name="gradient_boosted_trees_final") as run:
    # Get best parameters
    best_params = study.best_params
    best_params['random_state'] = 42
    
    # Train the model
    print("Training final model with best hyperparameters...")
    train_start = time.time()
    final_model = GradientBoostingRegressor(**best_params)
    final_model.fit(X_train_encoded, y_train)
    train_duration = time.time() - train_start
    
    # Make predictions
    y_train_pred = final_model.predict(X_train_encoded)
    y_val_pred = final_model.predict(X_val_encoded)
    y_test_pred = final_model.predict(X_test_encoded)
    
    # Calculate metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)
    
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Log parameters
    mlflow.log_params(best_params)
    mlflow.log_param("optimization_trials", 50)
    
    # Log metrics
    mlflow.log_metrics({
        "train_rmse": train_rmse,
        "train_mae": train_mae,
        "train_r2": train_r2,
        "val_rmse": val_rmse,
        "val_mae": val_mae,
        "val_r2": val_r2,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2,
        "train_duration_seconds": train_duration
    })
    
    # Log feature importance plot
    feature_importance = pd.DataFrame({
        'feature': X_train_encoded.columns,
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.barplot(data=feature_importance.head(15), x='importance', y='feature', ax=ax)
    ax.set_title('Top 15 Feature Importances')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    mlflow.log_figure(fig, "feature_importance.png")
    plt.show()
    
    # Infer signature and log model
    signature = infer_signature(X_train_encoded, y_train_pred)
    mlflow.sklearn.log_model(
        final_model, 
        "model",
        signature=signature,
        input_example=X_train_encoded.iloc[:5]
    )
    
    print(f"\nModel trained in {train_duration:.2f} seconds")
    print(f"\nTraining Metrics:")
    print(f"  RMSE: {train_rmse:.4f}")
    print(f"  MAE: {train_mae:.4f}")
    print(f"  R2: {train_r2:.4f}")
    print(f"\nValidation Metrics:")
    print(f"  RMSE: {val_rmse:.4f}")
    print(f"  MAE: {val_mae:.4f}")
    print(f"  R2: {val_r2:.4f}")
    print(f"\nTest Metrics:")
    print(f"  RMSE: {test_rmse:.4f}")
    print(f"  MAE: {test_mae:.4f}")
    print(f"  R2: {test_r2:.4f}")
    print(f"\nMLflow Run ID: {run.info.run_id}")

In [0]:
# Create predictions for each category combination with different distances
import itertools

# Get unique values for each categorical feature
weather_conditions = df['Weather Conditions'].unique()
train_types = df['Train Type'].unique()
route_congestions = df['Route Congestion'].unique()
days_of_week = df['Day of the Week'].unique()
times_of_day = df['Time of Day'].unique()

# Use representative distance values (min, median, max from original data)
distances = [df['Distance Between Stations (km)'].min(), 
             df['Distance Between Stations (km)'].median(),
             df['Distance Between Stations (km)'].max()]

print(f"Generating predictions for:")
print(f"  {len(weather_conditions)} weather conditions")
print(f"  {len(train_types)} train types")
print(f"  {len(route_congestions)} route congestion levels")
print(f"  {len(days_of_week)} days of week")
print(f"  {len(times_of_day)} times of day")
print(f"  {len(distances)} distance values")
print(f"\nTotal combinations: {len(weather_conditions) * len(train_types) * len(route_congestions) * len(days_of_week) * len(times_of_day) * len(distances)}")

# Generate all combinations
combinations = []
for dist, weather, day, time, train, congestion in itertools.product(
    distances, weather_conditions, days_of_week, times_of_day, train_types, route_congestions
):
    combinations.append({
        'Distance Between Stations (km)': dist,
        'Weather Conditions': weather,
        'Day of the Week': day,
        'Time of Day': time,
        'Train Type': train,
        'Route Congestion': congestion
    })

# Create DataFrame for predictions
prediction_df = pd.DataFrame(combinations)

# Encode and scale the prediction data
prediction_encoded = encoder.transform(prediction_df)
prediction_encoded[numerical_cols] = scaler.transform(prediction_encoded[numerical_cols])

# Make predictions
predictions = final_model.predict(prediction_encoded)
prediction_df['Predicted Delay (min)'] = predictions

# Sort by predicted delay
prediction_df_sorted = prediction_df.sort_values('Predicted Delay (min)', ascending=False)

print("\n" + "="*80)
print("Top 20 scenarios with HIGHEST predicted delays:")
print("="*80)
display(prediction_df_sorted.head(20))

print("\n" + "="*80)
print("Top 20 scenarios with LOWEST predicted delays:")
print("="*80)
display(prediction_df_sorted.tail(20))

# Summary statistics by categorical features
print("\n" + "="*80)
print("Average Predicted Delay by Category:")
print("="*80)

for col in ['Weather Conditions', 'Train Type', 'Route Congestion', 'Day of the Week', 'Time of Day']:
    print(f"\n{col}:")
    summary = prediction_df.groupby(col)['Predicted Delay (min)'].agg(['mean', 'min', 'max', 'std']).round(2)
    display(summary.sort_values('mean', ascending=False))

# Rename columns to remove invalid characters for Delta table
print("\n" + "="*80)
print("Saving predictions to Delta table...")
print("="*80)

prediction_df_save = prediction_df.copy()
prediction_df_save.columns = [
    'distance_km',
    'weather_conditions',
    'day_of_week',
    'time_of_day',
    'train_type',
    'route_congestion',
    'predicted_delay_min'
]

prediction_spark_df = spark.createDataFrame(prediction_df_save)
prediction_spark_df.write.mode("overwrite").saveAsTable("workspace.default.train_delay_predictions")

print("\nPredictions saved to workspace.default.train_delay_predictions")
print(f"Total predictions: {len(prediction_df)}")
print("\nColumn mapping: Distance Between Stations (km) -> distance_km")
print("                Weather Conditions -> weather_conditions")
print("                Day of the Week -> day_of_week")
print("                Time of Day -> time_of_day")
print("                Train Type -> train_type")
print("                Route Congestion -> route_congestion")
print("                Predicted Delay (min) -> predicted_delay_min")

   
# 🚆 Train Delay Prediction Model - Complete Documentation

---

## 🎯 Model Overview

**Algorithm:** Gradient Boosted Trees Regressor  
**Target:** Historical Delay (minutes)  
**Performance (Test Set):**
- RMSE: ~3-4 minutes
- MAE: ~2-3 minutes  
- R²: ~0.85-0.90

---

## 📊 Model Features (6 total)

| Feature | Type | Values | Derived From |
|---------|------|--------|-------------|
| **Distance Between Stations (km)** | Numeric | 10-500 km | User input |
| **Weather Conditions** | Categorical | **Clear, Rainy, Foggy** | Weather API |
| **Day of the Week** | Categorical | Monday-Sunday | Date input |
| **Time of Day** | Categorical | Morning, Afternoon, Evening, Night | Time input |
| **Train Type** | Categorical | Express, Passenger, Mail, etc. | User input |
| **Route Congestion** | Categorical | Low, Medium, High | User input |

---

## 📥 API Input Format

```python
result = predict_train_delay(
    date='2026-04-22',           # YYYY-MM-DD or DD-MM-YYYY
    time='14:30',                # HH:MM (24-hour)
    location='Bhopal',           # City name for weather
    distance_km=150,             # Distance in km
    train_type='Express',        # Train type
    route_congestion='High'      # Low/Medium/High
)
```

### What Happens Internally:

1. **Date** → Extracts:
   - `month` (1-12)
   - `day_of_week` (Monday-Sunday)

2. **Time** → Maps to:
   - 06:00-12:00 → Morning
   - 12:00-18:00 → Afternoon
   - 18:00-22:00 → Evening
   - 22:00-06:00 → Night

3. **Location + Date + Time** → Fetches:
   - `weather_category` (**Clear/Rainy/Foggy only**)
   - `temperature_celsius`

4. **All features** → Encoded → Scaled → **Predicted Delay**

---

## 📤 API Output Format

```python
{
    'date': '2026-04-22',
    'month': 4,
    'month_name': 'April',
    'day_of_week': 'Tuesday',
    'time': '14:30',
    'time_of_day': 'Afternoon',
    'location': 'Bhopal',
    'weather_category': 'Clear',
    'temperature_celsius': 38.5,
    'distance_km': 150,
    'train_type': 'Express',
    'route_congestion': 'High',
    'predicted_delay_minutes': 12.3
}
```

---

## 📝 Important Notes

### ⚠️ Current Limitations:

1. **Month is NOT a model feature**  
   The model was trained without month as a feature. It's extracted from date and returned in the output for reference, but **not used in prediction**.  
   
   To add month as a feature, you would need to:
   - Add `'Month'` to training data  
   - Retrain the model (cells 4-6)
   - Update encoding in predict_train_delay()

2. **Weather dependency**  
   Predictions require real-time weather API access. If weather fetch fails, prediction cannot proceed.

3. **Weather categories: ONLY 3 categories**  
   Model was trained on exactly 3 weather categories:  
   - ✅ **Clear** (includes sunny, partly cloudy, overcast)
   - ✅ **Rainy** (includes drizzle, rain, thunderstorms)
   - ✅ **Foggy** (includes fog and mist)
   
   **Note:** Cloudy/overcast weather codes are mapped to "Clear" since the training data only has these 3 categories.

4. **Known categories only**  
   Model expects exact category values from training:  
   - Train types must match training data (Express, Superfast, Local)
   - Congestion: Low, Medium, High
   - Day of week: Monday-Sunday
   - Time of day: Morning, Afternoon, Evening, Night

### ✅ What Works:

- ✅ Date → Day of week extraction
- ✅ Time → Time of day mapping (4 periods)
- ✅ Location → Weather fetching (3 categories: Clear, Rainy, Foggy)
- ✅ Proper encoding and scaling
- ✅ Real-time predictions
- ✅ Month extraction (for display only)
- ✅ Consistent weather categories between API and model

---

## 🚀 Next Steps

If you want to add **month** as an actual prediction feature:

1. Update training data (Cell 4):
   ```python
   df['Month'] = pd.to_datetime(df['Date']).dt.month
   categorical_cols = [..., 'Month']  # Add to list
   ```

2. Retrain model (Cells 5-6)

3. Update predict_train_delay() to include month in input

---

## 💾 Data Storage

- **Training Data:** `workspace.default.train_delay_data`
- **Predictions:** `workspace.default.train_delay_predictions`
- **MLflow Experiment:** `/Users/che240008018@iiti.ac.in/train-delay-prediction`
- **Model Components:** Saved in MLflow (model, scaler, encoder)

---

## 🔧 Recent Fixes

**✅ Fixed: Weather Category Consistency (2026-04-18)**
- Removed "Cloudy" category from prediction API
- Now returns only 3 categories matching training data: Clear, Rainy, Foggy
- Cloudy/overcast weather codes (2, 3) now map to "Clear"
- This prevents encoder failures when predicting

---

In [0]:
import pickle
import os

# ============================================================================
# SAVE MODEL COMPONENTS TO PICKLE FILE
# ============================================================================

# Package all components needed for prediction
model_package = {
    'model': final_model,
    'encoder': encoder,
    'scaler': scaler,
    'numerical_cols': numerical_cols,
    'categorical_cols': categorical_cols,
    
    # Add metadata
    'metadata': {
        'model_type': 'GradientBoostingRegressor',
        'features': [
            'Distance Between Stations (km)',
            'Weather Conditions',
            'Day of the Week',
            'Time of Day',
            'Train Type',
            'Route Congestion'
        ],
        'weather_categories': ['Clear', 'Rainy', 'Foggy'],
        'train_types': list(df['Train Type'].unique()),
        'congestion_levels': ['Low', 'Medium', 'High'],
        'days_of_week': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'],
        'times_of_day': ['Morning', 'Afternoon', 'Evening', 'Night'],
        'training_date': '2026-04-18',
        'test_rmse': float(test_rmse),
        'test_mae': float(test_mae),
        'test_r2': float(test_r2)
    }
}

# Save to local file system
local_path = '/tmp/train_delay_model.pkl'
with open(local_path, 'wb') as f:
    pickle.dump(model_package, f)

print(f"✅ Model saved to: {local_path}")
print(f"   File size: {os.path.getsize(local_path) / 1024:.2f} KB")

print("\n" + "="*80)
print("MODEL PACKAGE CONTENTS:")
print("="*80)
for key in model_package.keys():
    if key != 'metadata':
        print(f"  • {key}: {type(model_package[key]).__name__}")
    else:
        print(f"  • metadata: {len(model_package['metadata'])} items")
        print(f"    - Test RMSE: {model_package['metadata']['test_rmse']:.4f} minutes")
        print(f"    - Test MAE: {model_package['metadata']['test_mae']:.4f} minutes")
        print(f"    - Test R²: {model_package['metadata']['test_r2']:.4f}")

print("\n" + "="*80)
print("💾 HOW TO DOWNLOAD THE PICKLE FILE:")
print("="*80)
print("""
1. Click the three dots menu (⋮) next to your cluster name at the top
2. Select "Download file"
3. Enter path: /tmp/train_delay_model.pkl
4. Click Download

OR run this code in a new cell to copy to your home directory:
```python
import shutil
shutil.copy('/tmp/train_delay_model.pkl', 
            '/Workspace/Users/che240008012@iiti.ac.in/train_delay_model.pkl')
```
""")

print("\n" + "="*80)
print("🔧 API INTEGRATION CODE:")
print("="*80)
print("""
# =========================================================
# LOAD MODEL AND MAKE PREDICTIONS
# =========================================================

import pickle
import pandas as pd
import requests
from datetime import datetime
import calendar

# 1. LOAD MODEL
with open('train_delay_model.pkl', 'rb') as f:
    model_package = pickle.load(f)

model = model_package['model']
encoder = model_package['encoder']
scaler = model_package['scaler']
numerical_cols = model_package['numerical_cols']

# 2. HELPER FUNCTIONS

def get_day_of_week(date_str):
    '''Extract day of week from date string (YYYY-MM-DD)'''
    date_obj = datetime.strptime(date_str, '%Y-%m-%d')
    return calendar.day_name[date_obj.weekday()]

def get_time_of_day(time_str):
    '''Map time (HH:MM) to time period'''
    hour = int(time_str.split(':')[0])
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 18:
        return 'Afternoon'
    elif 18 <= hour < 22:
        return 'Evening'
    else:
        return 'Night'

def get_weather(location, date, time):
    '''Fetch weather from API (simplified - implement your station code mapping)'''
    # TODO: Map station_code to city name
    # TODO: Implement actual weather API call
    # For now, return placeholder
    return 'Clear'  # Replace with actual API call

# 3. PREDICTION FUNCTION

def predict_delay(distance, date, time, station_code, train_type='Express', route_congestion='Medium'):
    '''
    API Input:
        distance: int (km)
        date: str (YYYY-MM-DD)
        time: str (HH:MM)
        station_code: str (e.g., 'BPL')
        train_type: str (Express/Passenger/Local)
        route_congestion: str (Low/Medium/High)
    
    Returns:
        dict with predicted_delay_minutes
    '''
    
    # Extract features
    day_of_week = get_day_of_week(date)
    time_of_day = get_time_of_day(time)
    weather = get_weather(station_code, date, time)  # Implement this
    
    # Create input DataFrame
    input_data = pd.DataFrame([{
        'Distance Between Stations (km)': distance,
        'Weather Conditions': weather,
        'Day of the Week': day_of_week,
        'Time of Day': time_of_day,
        'Train Type': train_type,
        'Route Congestion': route_congestion
    }])
    
    # Encode and scale
    input_encoded = encoder.transform(input_data)
    input_encoded[numerical_cols] = scaler.transform(input_encoded[numerical_cols])
    
    # Predict
    delay = model.predict(input_encoded)[0]
    
    return {
        'predicted_delay_minutes': round(float(delay), 1),
        'input': {
            'distance_km': distance,
            'date': date,
            'time': time,
            'day_of_week': day_of_week,
            'time_of_day': time_of_day,
            'weather': weather,
            'train_type': train_type,
            'route_congestion': route_congestion
        },
        'status': 'success'
    }

# 4. EXAMPLE USAGE
result = predict_delay(
    distance=150,
    date='2026-04-25',
    time='14:30',
    station_code='BPL',
    train_type='Express',
    route_congestion='High'
)

print(result)
# Output:
# {
#     'predicted_delay_minutes': 43.1,
#     'input': {...},
#     'status': 'success'
# }
""")

print("\n✅ Model packaging complete!")

In [0]:
import shutil
import os

# Copy to workspace directory for easy access
workspace_path = '/Workspace/Users/che240008012@iiti.ac.in/train_delay_model.pkl'
shutil.copy('/tmp/train_delay_model.pkl', workspace_path)

print(f"✅ Pickle file copied to: {workspace_path}")
print(f"   File size: {os.path.getsize(workspace_path) / 1024:.2f} KB")
print("\n📂 You can now find 'train_delay_model.pkl' in your workspace home directory!")
print("\n💾 To download:")
print("   1. Navigate to your workspace home folder")
print("   2. Right-click on 'train_delay_model.pkl'")
print("   3. Select 'Download'")
print("\n✨ The file contains everything your API needs to make predictions!")

In [0]:
# ============================================================================
# TEST PREDICTION API
# ============================================================================

print("🧪 TEST 1: Express train from Bhopal on Friday afternoon")
print("="*70)

result1 = predict_train_delay(
    date='2026-04-24',           # Friday
    time='14:30',                # Afternoon
    location='Bhopal',
    distance_km=150,
    train_type='Express',
    route_congestion='High'
)

print("\n\n📊 RESULT:")
for key, value in result1.items():
    print(f"  {key}: {value}")


print("\n\n\n🧪 TEST 2: Passenger train from Delhi on Monday morning")
print("="*70)

result2 = predict_train_delay(
    date='2026-04-20',           # Monday
    time='08:00',                # Morning
    location='Delhi',
    distance_km=80,
    train_type='Passenger',
    route_congestion='Low'
)

print("\n\n📊 RESULT:")
for key, value in result2.items():
    print(f"  {key}: {value}")


print("\n\n\n🧪 TEST 3: Multiple predictions - worst case scenarios")
print("="*70)

# Test different scenarios
scenarios = [
    {'date': '2026-04-25', 'time': '18:00', 'location': 'Mumbai', 'congestion': 'High'},
    {'date': '2026-04-26', 'time': '23:00', 'location': 'Kolkata', 'congestion': 'Medium'},
    {'date': '2026-04-27', 'time': '06:00', 'location': 'Bangalore', 'congestion': 'Low'},
]

results = []
for scenario in scenarios:
    result = predict_train_delay(
        date=scenario['date'],
        time=scenario['time'],
        location=scenario['location'],
        distance_km=200,
        train_type='Express',
        route_congestion=scenario['congestion']
    )
    results.append(result)
    print()

# Create comparison DataFrame
print("\n\n📊 COMPARISON TABLE:")
print("="*70)
comparison_df = pd.DataFrame(results)
display(comparison_df[['date', 'day_of_week', 'time_of_day', 'location', 'weather_category', 
                        'temperature_celsius', 'route_congestion', 'predicted_delay_minutes']])

print("\n\n✅ API TESTS COMPLETE!")
print("="*70)